# Extraction PDF avec PyPDF → Parquet

Ce notebook lit tous les PDFs des dossiers `pdf_to_extract/` et `pdf_already_extracted/`,  
extrait leur contenu texte avec **pypdf**, et enregistre le tout dans `resultats.parquet`  
dans une colonne `contentpypdf`.

---

**Comportement :**
- Si `resultats.parquet` existe déjà, la colonne `contentpypdf` est ajoutée/mise à jour.
- Si un PDF n'est pas encore dans le parquet, une nouvelle ligne est créée.
- Le fichier parquet est écrasé avec les données mises à jour.

In [ ]:
import pypdf
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

# ── Paramètres ────────────────────────────────────────────────────────────────
PDF_FOLDERS = [
    Path("pdf_to_extract"),
    Path("pdf_already_extracted"),
]
PARQUET_PATH = Path("resultats.parquet")
COLUMN_NAME  = "contentpypdf"

# ── Collecte de tous les PDFs ─────────────────────────────────────────────────
all_pdfs = []
for folder in PDF_FOLDERS:
    if folder.exists():
        all_pdfs.extend(sorted(folder.glob("*.pdf")))

print(f"{len(all_pdfs)} PDF(s) trouvé(s)")

In [ ]:
# ── Extraction du texte avec pypdf ────────────────────────────────────────────
def extract_text_pypdf(pdf_path: Path) -> str:
    """Extrait tout le texte d'un PDF page par page avec pypdf."""
    try:
        reader = pypdf.PdfReader(str(pdf_path))
        pages_text = []
        for page in reader.pages:
            text = page.extract_text()
            if text:
                pages_text.append(text)
        return "\n".join(pages_text)
    except Exception as e:
        print(f"  ⚠ Erreur sur {pdf_path.name} : {e}")
        return ""


results = []
for pdf_path in tqdm(all_pdfs, desc="Extraction"):
    text = extract_text_pypdf(pdf_path)
    results.append({"pdf_name": pdf_path.name, COLUMN_NAME: text})

df_new = pd.DataFrame(results)
print(f"\nExtraction terminée — {len(df_new)} ligne(s)")

In [ ]:
# ── Mise à jour du parquet ────────────────────────────────────────────────────
if PARQUET_PATH.exists():
    df_existing = pd.read_parquet(PARQUET_PATH)
    print(f"Parquet existant : {len(df_existing)} ligne(s), colonnes : {df_existing.columns.tolist()}")

    # Supprime l'ancienne colonne si elle existe déjà
    if COLUMN_NAME in df_existing.columns:
        df_existing = df_existing.drop(columns=[COLUMN_NAME])

    # Fusionne sur pdf_name (outer join pour inclure les nouveaux PDFs)
    df_out = df_existing.merge(df_new[["pdf_name", COLUMN_NAME]], on="pdf_name", how="outer")
else:
    print("Aucun parquet existant — création d'un nouveau fichier.")
    df_out = df_new

df_out.to_parquet(PARQUET_PATH, index=False)
print(f"\n✓ Parquet sauvegardé : {PARQUET_PATH}")
print(f"  Lignes : {len(df_out)}  |  Colonnes : {df_out.columns.tolist()}")
df_out[["pdf_name", COLUMN_NAME]].head()